# Multimodal LLMs & File Manipulation with LangChain

Modern frontier LLMs (such as Google Gemini 2.5 Flash/Pro and OpenAI GPT-4o) are natively multimodal. They can process interleaved text, images, audio, and documents (PDFs) within a single prompt context.

In LangChain, multimodal inputs are constructed using standard `HumanMessage` objects containing a list of structured content blocks:
* `{"type": "text", "text": "..."}`
* `{"type": "image_url", "image_url": {"url": "..."}}` (supports both public web URLs and Base64-encoded data URIs `data:image/png;base64,...`)

### Media & File Storage (`docs/`):
All media, logs, and input/output documents are stored in the organized `docs/` directory. All file paths in this notebook begin with `'docs/'`.

## 1. Environment Setup & Model Initialization

We load API keys from `.env` and initialize a multimodal-capable model such as `gemini-2.5-flash` or `gpt-4o`. We also verify the `docs/` folder exists.

In [ ]:
import os
import base64
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

# Initialize Gemini 2.5 Flash as the primary multimodal model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

# Ensure docs directory exists
docs_dir = Path("docs")
docs_dir.mkdir(parents=True, exist_ok=True)

print("Multimodal Model Initialized:", llm.model)
print("Files & Media Directory:", docs_dir.resolve())

## 2. Remote Image Input via Public URL

To pass an image hosted on the web, provide the direct image URL inside the `image_url` dictionary payload. The model will fetch and inspect the visual features.

In [ ]:
# Analyzing a remote diagram via public URL
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8b/House_sparrow_male_-_Galicia.jpg/640px-House_sparrow_male_-_Galicia.jpg"

message = HumanMessage(
    content=[
        {"type": "text", "text": "Describe the bird in this image, identify its probable species, and note any distinct plumage characteristics."},
        {"type": "image_url", "image_url": {"url": image_url}}
    ]
)

response = llm.invoke([message])
print("Visual Analysis Response:")
print(response.content)

## 3. Passing an Existing Local Image (`docs/image.png`) to the Model

For local files stored on disk (such as `docs/image.png`), we read the binary bytes, encode them into a Base64 string, and construct a standard Data URI: `data:image/png;base64,{b64_string}`.

The Data URI is passed in a `HumanMessage` along with textual instructions.

In [ ]:
def encode_image_to_base64(image_path: str) -> str:
    """Reads a local image file and returns a base64 encoded string."""
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# Path to the existing image in docs/
image_path = "docs/image.png"

# Encode local image to base64
b64_image = encode_image_to_base64(image_path)

# Build multimodal message with data URI
local_image_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Analyze this system architecture diagram from 'docs/image.png'. Identify each service, its port, latency/status details, and explain the data flow between them in bullet points."
        },
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{b64_image}"}
        }
    ]
)

response_local = llm.invoke([local_image_message])
print(f"--- Analysis of Local Image ({image_path}) ---\n")
print(response_local.content)

## 4. Multi-Image & Mixed Modality Prompting

You can pass multiple images in a single `HumanMessage` to perform visual diffing, UI redesign critiques, or compare charts across quarters.

In [ ]:
# Multi-image comparison example
url1 = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Cat_August_2010-4.jpg/320px-Cat_August_2010-4.jpg"
url2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg"

compare_message = HumanMessage(
    content=[
        {"type": "text", "text": "Compare these two feline photos. Highlight differences in posture, lighting, and focal depth in bullet points."},
        {"type": "image_url", "image_url": {"url": url1}},
        {"type": "image_url", "image_url": {"url": url2}}
    ]
)

comparison_response = llm.invoke([compare_message])
print("Multi-Image Comparative Analysis:")
print(comparison_response.content)

## 5. File Ingestion & Persistence: Reading from `docs/` and Writing Response to `docs/`

In production pipelines, LLMs read raw files from storage (`docs/system_activity.log`), perform automated reasoning/diagnosis, and persist their generated insights to an output file (`docs/incident_report.md`).

In [ ]:
from pathlib import Path

# Step 1: Ingest input file from docs/
input_log_path = Path("docs/system_activity.log")
log_content = input_log_path.read_text(encoding="utf-8")
print(f"Read {len(log_content)} characters from: {input_log_path}")

# Step 2: Pass file content to LLM for root-cause diagnosis & action plan
prompt = f"""
You are an expert site reliability engineer. Analyze the following system activity log from '{input_log_path}':

```
{log_content}
```

Provide a structured incident report containing:
1. Executive Summary of critical incidents.
2. Root-Cause Analysis (identifying affected services, timestamps, and error codes).
3. Recommended Immediate Mitigation Steps.
"""

diagnosis = llm.invoke(prompt)
print("\n--- Diagnostic Report Generated by LLM ---")
print(diagnosis.content)

# Step 3: Write the LLM response to an output file in docs/
output_report_path = Path("docs/incident_report.md")
output_report_path.write_text(diagnosis.content, encoding="utf-8")
print(f"\nSuccessfully saved analysis report to: {output_report_path}")

## 6. Verifying Stored Files in `docs/`

Let's inspect all media and document files organized inside the `docs/` folder.

In [ ]:
# List all media and document files currently in docs/
print("Contents of 'docs/' directory:")
for file_item in sorted(Path("docs").iterdir()):
    size_kb = file_item.stat().st_size / 1024
    print(f" - {file_item.name:<28} ({size_kb:.2f} KB)")